# Lunar Occultation Experiment

Aaron Parsons

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from matplotlib.gridspec import GridSpec
import os
import tqdm

import healpy
from astropy.time import Time
from astropy.coordinates import get_sun, get_body, Galactic
import astropy.units as u

from eigsep_sim.healpix import HPM
from eigsep_sim.coord import rot_m
from eigsep_sim.const import R_MOON, R_SUN, R_EARTH
from eigsep_sim.beam import Beam, BEAM_NPZ
from eigsep_sim.models import T21cmModel
from eigsep_sim.sky import SkyModel
from eigsep_sim.utils import moon_surface_distance, moon_reflect_vector, reflectivity, sample_disk
from eigsep_sim.src import disc_overlap_fraction
from eigsep_sim.lunar_orbit import LunarOrbit

DTYPE = 'float32'
PLOT = True
CMAP = 'plasma'
%matplotlib inline

In [ ]:
# Load frequencies from the beam file, then build the Beam object.
_bm_npz = np.load(BEAM_NPZ)
freqs = _bm_npz['freqs'].astype(DTYPE)
NCHAN = freqs.size

# peak_normalize=False preserves relative frequency response for Jy/K bookkeeping.
beam = Beam(freqs, peak_normalize=False)
nside_beam = beam._nside

# Beam solid angle per frequency channel (for Jy ↔ K conversion)
from scipy.constants import c as C, k as k_B
Jy_mks = 1e-26
dOmega_beam = 4 * np.pi / healpy.nside2npix(nside_beam)
Omega_beam = dOmega_beam * np.sum(beam.map, axis=0)  # (nfreq,)
K2Jy = 2 * k_B * (freqs / C)**2 * Omega_beam / Jy_mks
Jy2K = 1 / K2Jy

In [ ]:
if PLOT:
    fig = plt.figure(figsize=(12, 6))
    ch = np.argmin(np.abs(freqs - 50e6))
    bm_plot = beam.map[:, ch]
    healpy.mollview(10 * np.log10(bm_plot / bm_plot.max()),
                    cmap=CMAP, min=-20, max=0, fig=fig, xsize=400)
    plt.show()

In [ ]:
## every visible pixel on the sky takes a direct path to the antenna, and it also takes an indirect path to each terrain point
## visible to the antenna, where it can bounce and enter the antenna. The path length difference is the radial distance
## from the antenna to the terrain point, plus the baseline distance from the antenna to the terrain point projected toward
## each sky point.
## This baseline is np.dot(r * crd[terrain], crd[sky])
#gnd_mask = ~np.isnan(r)
#r_vec = np.array(healpy.pix2vec(nside_full, np.arange(r.size)), dtype='float32')
#bls = r_vec[:, gnd_mask] * r[None, gnd_mask] # only terrain pixels
#bl_proj = np.dot(crd[:, sky_mask].T, bls) + r[None, gnd_mask]
#bl_proj = bl_proj.astype('float32')
#df = np.float32(freqs[1] - freqs[2])
#
## average T over all sky reflections; could put a scattering beam in here
#T_scat = np.empty((np.sum(gnd_mask), freqs.size))
#for i in tqdm.tqdm(range(freqs.size)):
#    T_scat[...,i] = np.mean(gamma[i] * T_sky[sky_mask, None, i] * np.cos(np.float32(2 * np.pi / C) * freqs[i] * bl_proj) * etu.pixel_delay_attenuation(bl_proj, df), axis=0)
#    T_scat[...,i] *= etu.pixel_coherence_angle_attenuation(r[gnd_mask], freqs[i], nside=nside_full)
#
#gsm_full = aipy.healpix.HealpixMap(nside=nside_full)
#gsm_full.from_alm(alm)
#gsm_full[0, np.sqrt(0.5), np.sqrt(0.5)] = 10
#T_sky_full = T_sky_spec[None, :] * gsm_full.map[:, None]
#T_gnd = 300 * np.ones_like(T_scat)
#T_sky_full[gnd_mask] = T_scat + T_gnd

In [ ]:
# Load 21cm cosmological signal model
t21cm_model = T21cmModel()
MODEL_INDEX = 120
T_21cm = t21cm_model(freqs, model_index=MODEL_INDEX).astype(DTYPE)

In [ ]:
if PLOT:
    fig, axes = plt.subplots(figsize=(8, 3))
    mdl_freqs = t21cm_model._freqs
    mdl_T = t21cm_model._mdls
    _ = axes.plot(mdl_freqs / 1e6, mdl_T.T, alpha=0.1)
    axes.set_title('Injected Model 21cm Signal')
    axes.set_ylabel('Temperature [K]')
    _ = axes.set_xlabel('Frequency [MHz]')
    plt.show()

In [ ]:
# Lunar surface reflectivity as a function of frequency
resistivity_ohm_m = 3e2  # Ohm·m, middle-of-the-road regolith estimate
gamma = reflectivity(freqs, resistivity_ohm_m).astype(DTYPE)

In [ ]:
if PLOT:
    plt.figure(figsize=(8,3))
    plt.plot(freqs / 1e6, gamma)
    plt.title('Reflection Coefficient')
    plt.xlabel('Frequency [MHz]')
    plt.ylabel('Fractional Power Reflected')
    plt.show()

In [ ]:
# Sky model: GSM2016 + bright point sources, galactic frame
date = "2024-11-20 12:00:00"  # UTC
t = Time(date)

NSIDE_SIM = 64
NPIX = healpy.nside2npix(NSIDE_SIM)
hpx_area = 4 * np.pi / NPIX

# SkyModel handles GSM loading and source injection internally.
# srcs=None removes point sources; use default list to include bright sources.
sky_model = SkyModel(freqs, nside=NSIDE_SIM)
Tsky = sky_model.map  # (NPIX, NCHAN), galactic frame, Kelvin

# Galactic-frame pixel unit vectors (used for rotation, masking, reflection)
x, y, z = healpy.pix2vec(NSIDE_SIM, np.arange(NPIX))
x, y, z = x.astype(DTYPE), y.astype(DTYPE), z.astype(DTYPE)
xyz = np.array([x, y, z], dtype=DTYPE)

In [ ]:
# Solar-system body positions relative to the Moon centre (galactic frame, metres)
moon_barypos = get_body('moon', t).transform_to(Galactic())
earth_barypos = get_body('earth', t).transform_to(Galactic())
sun_barypos = get_sun(t).transform_to(Galactic())

sun_pos = (sun_barypos.cartesian - moon_barypos.cartesian)
earth_pos = (earth_barypos.cartesian - moon_barypos.cartesian)
sun_pos_gal = np.array(sun_pos.xyz.to(u.m).value, dtype=DTYPE)    # (3,) metres, galactic frame
earth_pos_gal = np.array(earth_pos.xyz.to(u.m).value, dtype=DTYPE)  # (3,) metres, galactic frame

d_sun = np.linalg.norm(sun_pos_gal)
d_earth = np.linalg.norm(earth_pos_gal)

# Extended disc masks using disc_overlap_fraction (replaces Monte-Carlo sampling)
sun_angular_radius = np.arctan2(R_SUN, d_sun)
sun_angular_area = np.pi * sun_angular_radius**2
sun_pix, sun_frac = disc_overlap_fraction(NSIDE_SIM, sun_pos_gal / d_sun, sun_angular_radius)
sun_mask = np.zeros(NPIX, dtype=DTYPE)
sun_mask[sun_pix] = (sun_frac * sun_angular_area / hpx_area).clip(0, 1)

earth_angular_radius = np.arctan2(R_EARTH, d_earth)
earth_angular_area = np.pi * earth_angular_radius**2
earth_pix, earth_frac = disc_overlap_fraction(NSIDE_SIM, earth_pos_gal / d_earth, earth_angular_radius)
earth_mask = np.zeros(NPIX, dtype=DTYPE)
earth_mask[earth_pix] = (earth_frac * earth_angular_area / hpx_area).clip(0, 1)

# Solar radio spectrum (Erickson et al. 1977)
T_sun = 1e6       # K at fq0_sun
fq0_sun = 65e6   # Hz
sun_spec_index = 0.3
sun_spec = T_sun * (freqs / fq0_sun)**sun_spec_index

# Earth thermal + FM contamination
T_earth = 300 * np.ones(NCHAN, dtype=DTYPE)
fm_band = (freqs > 88e6) & (freqs < 109e6)
T_earth[fm_band] = np.random.uniform(0, 2000, size=fm_band.sum())

# Composite sky map: GSM + 21cm + Sun + Earth
tot_sky_map = Tsky + T_21cm[None, :]
tot_sky_map = (1 - sun_mask[:, None]) * tot_sky_map + sun_mask[:, None] * sun_spec[None, :]
tot_sky_map = (1 - earth_mask[:, None]) * tot_sky_map + earth_mask[:, None] * T_earth[None, :]

In [ ]:
if PLOT:
    fig = plt.figure(figsize=(12, 8))
    healpy.mollview(np.log10(tot_sky_map[:, 48]), max=4, min=2, cmap=CMAP)
    plt.show()

In [ ]:
T_moon_base = 200   # K, baseline lunar surface temperature
T_moon_lit = 100    # K, additional temperature on sun-lit side

NORBIT = 256
NSPIN = 32
PLOT_CH = np.argmin(np.abs(freqs - 80e6))

rot_spin_vec = np.array([0, 1, 0], dtype=DTYPE)
rot_orbit_vec = np.array([0, 0, 1], dtype=DTYPE)

# LunarOrbit tracks spacecraft position and spin in the galactic frame.
orbit = LunarOrbit(
    altitude=100e4,          # 1000 km above the lunar surface
    rot_orbit_vec=rot_orbit_vec,
    rot_spin_vec=rot_spin_vec,
    start_pos=[1, 0, 0],
)
d_moon = orbit.orbital_radius  # metres from Moon centre

# Pre-compute beam response for each spin angle (does not change with orbital phase)
beam_resp = np.empty((NSPIN, NPIX, NCHAN), dtype=DTYPE)
beam_now_center = np.empty((NSPIN, 3), dtype=DTYPE)
beam_start_center = np.array([0, 0, 1], dtype=DTYPE)

for cnt2, th_spin in enumerate(np.linspace(0, 2 * np.pi, NSPIN, endpoint=False)):
    _rot_m = rot_m(th_spin, rot_spin_vec).astype(DTYPE)
    beam_now_center[cnt2] = _rot_m @ beam_start_center
    # Rotate sky pixel directions into beam frame and look up beam response
    bx, by, bz = _rot_m @ xyz
    beam_resp[cnt2] = np.asarray(beam[np.array([bx, by, bz], dtype=DTYPE)])

beam_norm = np.sum(beam_resp, axis=1)  # (NSPIN, NCHAN)
T_obs = np.empty((NORBIT, NSPIN, NCHAN), dtype=DTYPE)

In [ ]:
moon_now_pos = np.empty((NORBIT, 3), dtype=DTYPE)

# Build a low-resolution sky HPM for reflected-sky lookups inside the main loop.
# Downsampling channel-by-channel is safer for large 2D maps.
NSIDE_LORES = 16
lores_sky_map = np.empty((healpy.nside2npix(NSIDE_LORES), NCHAN), dtype=DTYPE)

sky_hpm = HPM(nside=NSIDE_SIM, interp=True)
sky_hpm.set_map(tot_sky_map.astype(DTYPE))
sky_hpm_lores = HPM(nside=NSIDE_LORES)
sky_hpm_lores.set_map(np.zeros((healpy.nside2npix(NSIDE_LORES),), dtype=DTYPE))

for ch in range(NCHAN):
    sky_hpm.map = tot_sky_map[:, ch]
    sky_hpm_lores.from_hpm(sky_hpm)
    lores_sky_map[:, ch] = sky_hpm_lores.map

sky_hpm.map = tot_sky_map  # restore 2D map
sky_hpm_lores.map = lores_sky_map * (NSIDE_LORES / NSIDE_SIM)**2  # rescale sum → mean
sky_hpm_lores.set_interpol(1)

In [ ]:
SAVE_PNG = False
pct_dilate = 0.01

for cnt1, th_orbit in tqdm.tqdm(
        enumerate(np.linspace(0, 2 * np.pi, NORBIT, endpoint=False)), total=NORBIT):

    # Spacecraft position from Moon centre in the galactic frame (metres)
    orbit.set_phases(th_orbit)
    moon_now_pos[cnt1] = orbit.spacecraft_position().astype(DTYPE)

    # Angle from each sky pixel to the Moon direction, and distance to Moon surface
    alpha = healpy.rotator.angdist(xyz, moon_now_pos[cnt1] / d_moon).astype(DTYPE)
    d_surface = moon_surface_distance(alpha, d_moon).astype(DTYPE)

    # Feather the Moon's edge at the pixel level
    moon_mask = np.where(np.isnan(d_surface), 0.0, 1.0).astype(DTYPE)
    edge = np.where(np.logical_and(
        ~np.isnan(moon_surface_distance(alpha, d_moon * (1 - pct_dilate))),
         np.isnan(moon_surface_distance(alpha, d_moon * (1 + pct_dilate)))
    ))[0]
    for px in edge:
        vec = xyz[:, px]
        samples = sample_disk(vec, 0.8 * np.sqrt(hpx_area), 1000)
        samples = samples[:, healpy.vec2pix(NSIDE_SIM, *samples) == px]
        _alpha = healpy.rotator.angdist(samples, moon_now_pos[cnt1] / d_moon).astype(DTYPE)
        _d_surface = moon_surface_distance(_alpha, d_moon).astype(DTYPE)
        moon_mask[px] = np.mean(np.where(np.isnan(_d_surface), 0, 1))

    # Lunar surface illumination (dot product of surface normal with Sun direction)
    on_moon = ~np.isnan(d_surface)
    vec_to_surface = xyz * d_surface
    lit_up = (
        np.dot(sun_pos_gal - moon_now_pos[cnt1], vec_to_surface - moon_now_pos[cnt1, :, None])
        / d_sun / R_MOON
    ).clip(0)

    # Blend: thermal emission + reflected sky on Moon surface
    moon_mask_refl = moon_mask[:, None] * (1 - gamma[None, :])
    T_moon = T_moon_base + T_moon_lit * np.nan_to_num(lit_up, 0)  # K

    _x, _y, _z = moon_reflect_vector(vec_to_surface[:, on_moon], moon_now_pos[cnt1])
    reflection = sky_hpm_lores[_x, _y, _z]

    sky_map_now = tot_sky_map.copy()
    sky_map_now[on_moon] = reflection
    _sky = (1 - moon_mask_refl) * sky_map_now + moon_mask_refl * T_moon[:, None]

    # Beam-weighted antenna temperature for each spin orientation
    beam_sky = beam_resp * _sky[None, ...]
    T_obs[cnt1] = np.sum(beam_sky, axis=1) / beam_norm

    if SAVE_PNG:
        for cnt2 in range(NSPIN):
            outfile = f'gsm_moon_reflect_{cnt1 * NSPIN + cnt2:04d}.png'
            if os.path.exists(outfile):
                continue
            fig = plt.figure(figsize=(10, 8))
            gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.1)
            axes = [fig.add_subplot(gs[0]), fig.add_subplot(gs[1])]
            plt.axes(axes[0])
            healpy.mollview(np.log10(beam_sky[cnt2, :, PLOT_CH]),
                            cmap=CMAP, min=1, max=4, xsize=400, fig=fig, hold=True)
            T_rel = T_obs / T_obs[:1, :1]
            axes[1].plot(freqs / 1e6, T_rel[:cnt1].reshape(-1, NCHAN).T, alpha=0.2)
            axes[1].plot(freqs / 1e6, T_rel[cnt1, :cnt2].reshape(-1, NCHAN).T, alpha=0.2)
            axes[1].plot(freqs / 1e6, T_rel[cnt1, cnt2], 'k')
            axes[1].set_xlabel('Frequency [MHz]')
            axes[1].set_ylabel('T / T_0')
            axes[1].set_ylim(0.7, 3)
            plt.savefig(outfile)
            plt.close('all')

In [ ]:
if False:
    np.savez('lunar_occultation_sim_v001.npz',
             T_obs=T_obs,
             date=date,
             freqs=freqs,
             srcs=['GSM', 'T21cm', 'FM', 'Sun', 'Earth'],
             T21cm_model_num=MODEL_INDEX,
             r_sun=R_SUN,
             sun_pos=sun_pos_gal,
             r_moon=R_MOON,
             moon_pos=moon_now_pos,
             beam_center=beam_now_center,
             rot_spin_vec=rot_spin_vec,
             rot_orbit_vec=rot_orbit_vec,
    )

In [ ]:
if PLOT:
    fig, axes = plt.subplots(nrows=2, sharex=True, figsize=(8, 6))
    T_flat = np.reshape(T_obs, (-1, NCHAN))
    T_mean = np.mean(T_flat, axis=0, keepdims=True)
    axes[0].semilogy(freqs / 1e6, T_flat.T, alpha=0.2)
    axes[0].plot(freqs[1:] / 1e6, np.abs(np.diff(T_flat, axis=1)).T, alpha=0.2)
    _ = axes[1].plot(freqs / 1e6, (T_flat / T_mean).T, alpha=0.2)
    axes[1].set_xlabel('Frequency [MHz]')
    axes[0].set_ylabel('Temperature [K]')
    axes[1].set_ylabel('T/<T>')
    plt.show()

In [ ]:
#T_var = T_flat / T_mean - 1
T_var = T_flat - T_mean
C = np.dot(T_var.T, T_var)
print(C.shape)
U, S, Vt = np.linalg.svd(C)
_P = np.ones_like(S)
_P[:10] = 0
P = np.einsum('ij,j,jk->ik', U, _P, Vt)
T_res = np.dot(P, T_flat[:10].T).T
#T_res = np.dot(P, T_var[:10].T).T
#plt.figure()
#plt.plot(freqs / 1e6, T_res.T, alpha=0.2)
#plt.plot(freqs / 1e6, T_flat[:10].T, alpha=0.2)
#plt.show()
#plt.figure()
#plt.semilogy(S)
#plt.semilogy(_P)
#plt.show()
plt.figure()
#plt.plot(freqs / 1e6, U[:,:3])
plt.plot(freqs / 1e6, np.dot(P, T_21cm))
plt.plot(freqs / 1e6, T_21cm)
plt.show()


In [ ]:
x = np.logspace(-6, 2, 1000)
plt.figure()
plt.plot(x * np.log(x))
plt.plot(np.log(x))
plt.plot(np.log(x+1e-5))
plt.show()